In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
from linearmodels.iv import IV2SLS

## Problem 2
### Part 2a

In [2]:
data = pd.read_excel("PS4data.xls")
data['consumption'] = data['real consumption of nondurables'] + data['real consumption of services']
data['consumption per capita'] = data['consumption'] / data['population']
data['consumption per capita'].head()

0    0.008824
1    0.008914
2    0.008910
3    0.008991
4    0.009021
Name: consumption per capita, dtype: float64

In [3]:
lags = data['consumption per capita'].shift(periods=[1, 2, 3, 4])
data_with_lags = pd.concat([data, lags], axis=1)

In [4]:
data_with_lags_1 = data_with_lags.iloc[4:, :]

model = sm.OLS(data_with_lags_1['consumption per capita'], sm.add_constant(data_with_lags_1.iloc[:, -4:]))
model_fit = model.fit()

r_matrix = np.identity(5)[2:, :]
print(model_fit.f_test(r_matrix=r_matrix).pvalue)

2.6364145736190264e-05


We reject Hall's Hypothesis that beta_2 = beta_3 = beta_4 = 0 at the 5% significance level since the p-value is less than 0.05.

### Part 2b
This setup from Campbell and Mankiw has an endogeneity issue since current income is a part of permanent income. Thus the remainder is correlated with the error term. This will lead to inconsistent estimate of gamma_1.

### Part 2c

In [5]:
for i in range(0, 6):
    data_with_lags[f'log change consumption per capita_lag{i}'] = np.log(data_with_lags['consumption per capita'].shift(i) / data_with_lags['consumption per capita'].shift(i + 1))
endog = data_with_lags['log change consumption per capita_lag0']

data_with_lags['real disposable income per capita'] = data_with_lags['real disposable income'] / data_with_lags['population']
data_with_lags['real disposable income per capita_1'] = data_with_lags['real disposable income per capita'].shift(periods=1)
exog = sm.add_constant(np.log(data_with_lags['real disposable income per capita'] / data_with_lags['real disposable income per capita_1']))

instruments = data_with_lags[[f'log change consumption per capita_lag{i}' for i in range(2, 6)]]

In [6]:
iv_model = IV2SLS(endog[6:], exog[['const']][6:], exog.iloc[:, 1][6:], instruments[6:])
iv_results = iv_model.fit(cov_type='robust')
print(iv_results.params)
print(iv_results.std_errors)
print(iv_results.pvalues)

const    0.002613
endog    0.436925
Name: parameter, dtype: float64
const    0.001157
endog    0.202555
Name: stderr, dtype: float64
const    0.023954
endog    0.031000
Name: pvalue, dtype: float64


gamma_0 is small, but statistically significant. Gamma_1 is 0.436 and statistically significant. Thus, with this 2SLS regression the estimated fraction of consumers that spend current income is 0.436.

### Part 2d

In [7]:
print(f'P-value of exogeneity test: {iv_results.wooldridge_regression.pval}')

P-value of exogeneity test: 0.28567257012283676


In [8]:
n = len(data_with_lags[6:])

W = np.column_stack([np.ones(n), instruments[6:].values])
error = iv_results.resids.values.reshape(-1, 1)

S = n * (error.T @ W @ np.linalg.inv(W.T @ W) @ W.T @ error) / (error.T @ error)
chi2_stat = S[0][0]

l, k = W.shape[1], 2
p_value = 1 - chi2.cdf(chi2_stat, df=l - k)

print(f'Sargan statistic: {chi2_stat}, p-value: {p_value}')

Sargan statistic: 7.738279229290149, p-value: 0.05174198783914097


Since the exogeneity test p-value is greater than 0.05, we fail to reject the null hypothesis that log change in real disposable income per capita is not exogenous. Since the Sargan test p-value is greater than 0.05, we fail to reject the null hypothesis that the instruments are not exogenous. We can conclude that the instruments are valid and the 2SLS results are consistent.

## Problem 3
### Part 3d

In [9]:
spdata = pd.read_excel('SP500Index.xlsx')
spdata.head()

,Date of Observation,Level of the S&P 500 Index
0,19600129,55.61
1,19600229,56.12
2,19600331,55.34
3,19600429,54.37
4,19600531,55.83


In [10]:
spdata['xt'] = np.log(spdata.iloc[:, 1] / spdata.iloc[0, 1])
xt = spdata['xt'].iloc[1:]
xt_1 = spdata['xt'].shift(1).iloc[1:]
T = len(xt)
delta_ml = spdata['xt'].iloc[-1] / T
sigma_ml = np.sqrt(np.sum((xt - delta_ml - xt_1) ** 2) / T)
print(f'MLE estimates: delta = {delta_ml}, sigma = {sigma_ml}')

MLE estimates: delta = 0.005548394684384916, sigma = 0.04218034607391596


The estimated drift for the SP500 is 0.5% every month with a volatility of 4%. So the drift is very small, but the volatility is quite high.